# Your First World Bank Indicator

Download World Development Indicators with `WorldBankData.jl`. There is **no API key**. `wdi()` returns a `Tables.jl` table with one row per country and year and one column per indicator.

In [ ]:
Pollis.packages("WorldBankData", "Tables", "Plots")

In [ ]:
# import Pkg; Pkg.add(["WorldBankData", "Tables", "Plots"])
using WorldBankData, Tables, Plots

## One Indicator, One Country

`NY.GDP.PCAP.KD` is GDP per capita in constant 2015 US dollars. In the result, the dots in the code become underscores.

In [ ]:
tbl = wdi("NY.GDP.PCAP.KD", "PT", 2000, 2023)

years = Int.(Tables.getcolumn(tbl, :year))
gdppc = Tables.getcolumn(tbl, :NY_GDP_PCAP_KD)
foreach((y, v) -> println(y, "  ", round(v; digits = 0)), last(years, 5), last(gdppc, 5))

## Several Countries

Pass a vector of ISO2 codes. The `iso2c` column identifies each country's rows.

In [ ]:
tbl = wdi("NY.GDP.PCAP.KD", ["PT", "ES", "IE"], 2000, 2023)

iso = Tables.getcolumn(tbl, :iso2c)
years = Int.(Tables.getcolumn(tbl, :year))
gdppc = Tables.getcolumn(tbl, :NY_GDP_PCAP_KD)

p = plot(xlabel = "year", ylabel = "GDP per capita (constant 2015 US\$)", legend = :topleft)
for c in unique(iso)
    rows = findall(==(c), iso)
    plot!(p, years[rows], gdppc[rows], label = c)
end
p

## Growth Over the Period

Missing years are `missing`, so skip them before computing growth.

In [ ]:
for c in unique(iso)
    rows = findall(i -> iso[i] == c && !ismissing(gdppc[i]), eachindex(iso))
    y0, y1 = extrema(years[rows])
    v0, v1 = gdppc[rows][argmin(years[rows])], gdppc[rows][argmax(years[rows])]
    println(c, ": ", round(100 * ((v1 / v0)^(1 / (y1 - y0)) - 1); digits = 2), "% a year from ", y0, " to ", y1)
end

## Next Steps

- Add `extra = true` to get the region and income group of each country.
- Try `SP.DYN.LE00.IN` (life expectancy) or `SL.UEM.TOTL.ZS` (unemployment).